In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

In [ ]:
url1 = "https://drive.google.com/uc?id=1ZikFl_hiLYBgxL8-TkE4R2bA7Us7OQT0"
url2 = "https://drive.google.com/uc?id=12vxL3YIKD-BAqLs3ZGL6fUcmfl_S_GsA"
url3 = "https://drive.google.com/uc?id=1C7d7NDfGKqqiMOMB8SpXVAybSp2JUq5C"

gdown.download(url1, "datatrain.csv", quiet=False)
gdown.download(url2, "datatest.csv", quiet=False)
gdown.download(url3, "submission.csv", quiet=False)

In [ ]:
import os, time, random
import pandas as pd
from yt_dlp import YoutubeDL

OUT_DIR = "df_train_videos"
os.makedirs(OUT_DIR, exist_ok=True)

ydl_opts = {
    "outtmpl": os.path.join(OUT_DIR, "%(id)s.%(ext)s"),
    "noplaylist": True,
    "ignoreerrors": True,
    "cookiefile": "cookies.txt",
    "concurrent_fragment_downloads": 1,
    "retries": 5,
    "no_warnings": True,
}

failed_rows = []   # simpan row yang gagal
success_count = 0

def download_video(url, index, row):
    global success_count
    try:
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            vid_id = info.get("id", None)
            ext = info.get("ext", "mp4")

            orig_path = os.path.join(OUT_DIR, f"{vid_id}.{ext}")
            new_name = f"[{index}]{vid_id}.{ext}"
            new_path = os.path.join(OUT_DIR, new_name)

            if os.path.exists(new_path):
                print(f"✅ Skip (already exists): {new_path}")
                return

            if not os.path.exists(orig_path):
                ydl.download([url])

            if os.path.exists(orig_path):
                os.rename(orig_path, new_path)
                print(f"📂 Saved as: {new_path}")
                success_count += 1
            else:
                print(f"⚠️ File not found after download: {orig_path}")
                failed_rows.append(row)

    except Exception as e:
        print(f"❌ Failed {url}: {e}")
        failed_rows.append(row)

# Loop dataset
for i, row in df_train.iterrows():
    url = row["video"].strip()
    print(f"[{i+1}/{len(df_train)}] Checking: {url}")
    download_video(url, i+1, row)
    time.sleep(random.uniform(1, 2))

print(f"✅ Success downloaded: {success_count}")
print(f"❌ Failed: {len(failed_rows)}")

# simpan DataFrame gagal
df_train_failed = pd.DataFrame(failed_rows)
print("df_train_failed dibuat:", df_train_failed.shape)


In [ ]:
import os, time, random
import pandas as pd
from yt_dlp import YoutubeDL

OUT_DIR = "df_test_videos"
os.makedirs(OUT_DIR, exist_ok=True)

ydl_opts = {
    "outtmpl": os.path.join(OUT_DIR, "%(id)s.%(ext)s"),
    "noplaylist": True,
    "ignoreerrors": True,
    "cookiefile": "cookies.txt",
    "concurrent_fragment_downloads": 1,
    "retries": 5,
    "no_warnings": True,
}

failed_rows = []   # simpan row yang gagal
success_count = 0

def download_video(url, index, row):
    global success_count
    try:
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            vid_id = info.get("id", None)
            ext = info.get("ext", "mp4")

            orig_path = os.path.join(OUT_DIR, f"{vid_id}.{ext}")
            new_name = f"[{index}]{vid_id}.{ext}"
            new_path = os.path.join(OUT_DIR, new_name)

            if os.path.exists(new_path):
                print(f"✅ Skip (already exists): {new_path}")
                return

            if not os.path.exists(orig_path):
                ydl.download([url])

            if os.path.exists(orig_path):
                os.rename(orig_path, new_path)
                print(f"📂 Saved as: {new_path}")
                success_count += 1
            else:
                print(f"⚠️ File not found after download: {orig_path}")
                failed_rows.append(row)

    except Exception as e:
        print(f"❌ Failed {url}: {e}")
        failed_rows.append(row)

# Loop dataset
for i, row in df_test.iterrows():
    url = row["video"].strip()
    print(f"[{i+1}/{len(df_test)}] Checking: {url}")
    download_video(url, i+1, row)
    time.sleep(random.uniform(1, 2))

print(f"✅ Success downloaded: {success_count}")
print(f"❌ Failed: {len(failed_rows)}")

# simpan DataFrame gagal
df_test_failed = pd.DataFrame(failed_rows)
print("df_test_failed dibuat:", df_test_failed.shape)


In [ ]:
OUT_DIR_TRAIN = "/content/drive/MyDrive/satria_data_2025/df_train_videos"

files_train = [f for f in os.listdir(OUT_DIR_TRAIN) if f.endswith(".mp4")]

df_train_video = pd.DataFrame({
    "filename": files_train,
    "path": [os.path.join(OUT_DIR_TRAIN, f) for f in files_train]
})

print(df_train_video.head())
print(f"\nTotal video terdownload (TRAIN): {len(df_train_video)}")

In [ ]:
OUT_DIR_TEST = "/content/drive/MyDrive/satria_data_2025/df_test_videos"

files_test = [f for f in os.listdir(OUT_DIR_TEST) if f.endswith(".mp4")]

df_test_video = pd.DataFrame({
    "filename": files_test,
    "path": [os.path.join(OUT_DIR_TEST, f) for f in files_test]
})

print(df_test_video.head())
print(f"\nTotal video terdownload (TEST): {len(df_test_video)}")

In [ ]:
df_train_video['id'] = df_train_video['filename'].str.extract(r"\[(\d+)\]").astype(int)

df_train_merged = pd.merge(
    df_train,
    df_train_video[['id', 'path']],
    on="id",
    how="left"
)

# isi NaN di kolom path dengan "unknown"
df_train_merged['path'] = df_train_merged['path'].fillna("unknown")

In [ ]:
df_test_video['id'] = df_test_video['filename'].str.extract(r"\[(\d+)\]").astype(int)

df_test_merged = pd.merge(
    df_test,
    df_test_video[['id', 'path']],
    on="id",
    how="left"
)

# isi NaN di kolom path dengan "unknown"
df_test_merged['path'] = df_test_merged['path'].fillna("unknown")

In [ ]:
df_train_merged.to_csv("df_test_merged.csv", index=False)

In [ ]:
df_test_merged.to_csv("df_test_merged.csv", index=False)